# KLASIFIKASI GENRE GAME BERDASARKAN SINOPSIS — VERSI PERBAIKAN
---
**Author:** *Muhammad Wira Widhana [24.55.2717]*

> **Perbaikan dari versi sebelumnya:**
> 1. **KNN metric** diubah dari `euclidean` → `cosine` (lebih tepat untuk TF-IDF sparse)
> 2. **Penanganan class imbalance** dengan RandomUnderSampler
> 3. **GridSearchCV** untuk mencari k optimal KNN
> 4. **Fitur gabungan** title + synopsis untuk signal genre lebih kuat
> 5. **Skenario S4** ditambahkan: title + synopsis + stopword + stemming
> 6. Semua metric KNN di K-Fold dan perbandingan skenario ikut diperbarui

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Library & Load Dataset

### Import library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import ast
import math
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
)

from IPython.display import display

In [ ]:
# Download NLTK data (jalankan sekali)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

english_stopwords = set(stopwords.words('english'))
stemmer = PorterStemmer()

## Load dan Memeriksa Data (Data Loading & EDA)

### Load Dataset

In [ ]:
# Load Dataset
file_path = '/content/drive/MyDrive/Thesis_s2/games_fixed.csv'
df_raw = pd.read_csv(file_path)

print("Ukuran awal dataset:", df_raw.shape)
df_raw.head()

> Membaca dataset `games_fixed.csv` dari Kaggle via Google Drive.

### Seleksi kolom (judul, sinopsis, genre)

In [ ]:
# Seleksi kolom: judul, sinopsis, genre
df = df_raw[['Name', 'About the game', 'Genres']].copy()

df.rename(columns={
    'Name': 'title',
    'About the game': 'synopsis',
    'Genres': 'genre_raw'
}, inplace=True)

print("Jumlah data setelah seleksi kolom:", df.shape)
df.sample(5)

In [ ]:
# Cek missing values
df.isna().sum()

In [ ]:
# Hapus baris yang tidak punya sinopsis atau genre
df = df.dropna(subset=['synopsis', 'genre_raw']).reset_index(drop=True)

df['genre_raw'] = df['genre_raw'].astype(str).str.strip()
df = df[~df['genre_raw'].isin(['[]', "['']", '[""]', '', 'nan'])].reset_index(drop=True)

print("Jumlah data setelah drop NA dan genre kosong:", df.shape)

## Normalisasi Genre

Mengikuti paper: 4 genre dan single-label (satu data = satu genre).
Genre target: **action, adventure, rpg, simulation**.

In [ ]:
target_genres = {'action', 'adventure', 'rpg', 'simulation'}

def parse_genre_list(genre_str):
    '''Parse string genre menjadi list python yang sudah dinormalisasi ke huruf kecil.'''
    if pd.isna(genre_str):
        return []
    if isinstance(genre_str, list):
        raw_list = genre_str
    else:
        text = str(genre_str).strip()
        try:
            raw_list = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            raw_list = [g.strip() for g in text.split(',')]
    return [g.strip().lower() for g in raw_list if isinstance(g, str)]

df['genre_list']     = df['genre_raw'].apply(parse_genre_list)
df['genre_filtered'] = df['genre_list'].apply(lambda lst: [g for g in lst if g in target_genres])
df['genre_count']    = df['genre_filtered'].apply(len)

df[['genre_raw', 'genre_list', 'genre_filtered', 'genre_count']].head()

In [ ]:
# Single-label: hanya ambil data dengan tepat satu genre target
df_single = df[df['genre_count'] == 1].copy().reset_index(drop=True)
df_single['genre_main'] = df_single['genre_filtered'].str[0]

print("Jumlah data setelah filter single-label:", df_single.shape)
df_single[['title', 'synopsis', 'genre_raw', 'genre_filtered', 'genre_main']].head(10)

## Pembersihan Data

In [ ]:
def basic_english_filter(text):
    if not isinstance(text, str):
        return False
    clean = re.sub(r'[^A-Za-z\s]', ' ', text)
    clean = re.sub(r'\s+', ' ', clean).strip()
    if len(clean.split()) < 3:
        return False
    return (len(clean) / max(len(text), 1)) >= 0.7

df_single['is_english'] = df_single['synopsis'].apply(basic_english_filter)
df_single = df_single[df_single['is_english']].drop(columns=['is_english']).reset_index(drop=True)
print("Jumlah data setelah filter bahasa Inggris:", df_single.shape)

In [ ]:
MIN_WORDS, MIN_CHARS = 10, 40
_syn = df_single['synopsis'].astype(str).str.strip()
word_count   = _syn.str.split().apply(len)
char_count   = _syn.str.len()

placeholder_re = re.compile(
    r'^(no description( available)?|coming soon|tba|tbd|n/?a|none|\.+|-+)$', re.I)
is_placeholder = _syn.apply(lambda s: bool(placeholder_re.match(s)))
too_short      = (word_count < MIN_WORDS) | (char_count < MIN_CHARS)
is_dup         = _syn.str.lower().duplicated(keep='first')

before = len(df_single)
df_single = df_single[~(is_placeholder | too_short | is_dup)].reset_index(drop=True)
print(f"Data sebelum: {before} -> sesudah: {len(df_single)} (dibuang {before - len(df_single)})")

## Analisis Distribusi Genre & Penanganan Class Imbalance

> **[PERBAIKAN]** Langkah baru: cek rasio distribusi. Jika kelas terbesar > 2× kelas terkecil,
> lakukan **RandomUnderSampling** agar setiap genre punya jumlah data yang seimbang.
> Ini mencegah model bias ke genre mayoritas (biasanya action) dan menaikkan F1 kelas minoritas.

In [ ]:
# Distribusi genre sebelum balancing
genre_counts_before = df_single['genre_main'].value_counts()
print("=== Distribusi Genre (Sebelum Balancing) ===")
print(genre_counts_before)
print(f"\nRasio max/min: {genre_counts_before.max() / genre_counts_before.min():.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = genre_counts_before.plot(kind='bar', color='steelblue', edgecolor='black', ax=axes[0])
axes[0].set_title('Distribusi Genre (Sebelum Balancing)')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Jumlah Game')
axes[0].tick_params(axis='x', rotation=45)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)

In [ ]:
# ── Penanganan Class Imbalance dengan RandomUnderSampling ──────────────────
# Undersample semua kelas ke jumlah kelas terkecil (strategi konservatif)
min_count = genre_counts_before.min()
print(f"Jumlah target per kelas: {min_count}")

df_balanced = (
    df_single
    .groupby('genre_main', group_keys=False)
    .apply(lambda x: x.sample(min_count, random_state=42))
    .reset_index(drop=True)
)

genre_counts_after = df_balanced['genre_main'].value_counts()
print("\n=== Distribusi Genre (Setelah Balancing) ===")
print(genre_counts_after)

ax2 = genre_counts_after.plot(kind='bar', color='seagreen', edgecolor='black', ax=axes[1])
axes[1].set_title('Distribusi Genre (Setelah Balancing)')
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Jumlah Game')
axes[1].tick_params(axis='x', rotation=45)
for p in ax2.patches:
    ax2.annotate(f'{int(p.get_height())}',
                 (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()
print(f"\nTotal data setelah balancing: {len(df_balanced)}")

## Preprocessing

Urutan mengikuti paper: `Case Folding → Cleansing → Stopword Removal → Stemming → Tokenization`

Skenario preprocessing:
- **S1**: Case folding + Cleansing + Stopword removal
- **S2**: Case folding + Cleansing + Stemming
- **S3**: Case folding + Cleansing + Stopword + Stemming *(sesuai paper)*
- **S4** *(baru)*: Title + Synopsis + S3 — menggabungkan sinyal judul game

In [ ]:
def case_folding(text):
    return str(text).lower()

def cleansing(text):
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def stopword_removal(text):
    return ' '.join(w for w in text.split() if w not in english_stopwords)

def stemming(text):
    return ' '.join(stemmer.stem(w) for w in text.split())

def tokenization(text):
    return word_tokenize(text)

def preprocess(text, use_stopword=True, use_stemming=True):
    '''Pipeline preprocessing sesuai urutan paper.'''
    text = case_folding(text)
    text = cleansing(text)
    if use_stopword:
        text = stopword_removal(text)
    if use_stemming:
        text = stemming(text)
    tokens = tokenization(text)
    return ' '.join(tokens)

In [ ]:
# Terapkan 4 skenario preprocessing pada data yang sudah balanced
df_balanced['synopsis_S1'] = df_balanced['synopsis'].apply(
    lambda x: preprocess(x, True, False))   # stopword only
df_balanced['synopsis_S2'] = df_balanced['synopsis'].apply(
    lambda x: preprocess(x, False, True))   # stemming only
df_balanced['synopsis_S3'] = df_balanced['synopsis'].apply(
    lambda x: preprocess(x, True, True))    # stopword + stemming (sesuai paper)

# S4: gabung title + synopsis, lalu S3
df_balanced['synopsis_S4'] = (
    df_balanced['title'].fillna('') + ' ' + df_balanced['synopsis']
).apply(lambda x: preprocess(x, True, True))

df_balanced[['genre_main', 'synopsis', 'synopsis_S3', 'synopsis_S4']].head()

> Skenario S3 (stopword + stemming) adalah skenario utama sesuai paper. S4 menambah judul game sebagai sinyal tambahan.

## Pembobotan TF-IDF

Data dibagi 80% latih : 20% uji (stratified). TF-IDF di-`fit` hanya pada data latih.

> **[PERBAIKAN]** `min_df=2` (dari 3) agar fitur kelas kecil tidak terbuang.

In [ ]:
TEXT_COL = 'synopsis_S3'
X_text = df_balanced[TEXT_COL]
y      = df_balanced['genre_main']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data latih:", X_train_text.shape[0])
print("Jumlah data uji  :", X_test_text.shape[0])
print("\nDistribusi y_train:")
print(pd.Series(y_train).value_counts())

In [ ]:
# TF-IDF — fit pada train saja, transform pada test
# [PERBAIKAN] min_df=2 (lebih inklusif untuk kelas kecil)
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,          # <-- diperbaiki dari 3
    max_df=0.95
)

X_train = tfidf.fit_transform(X_train_text)
X_test  = tfidf.transform(X_test_text)

print("Dimensi matriks TF-IDF latih:", X_train.shape)
print("Dimensi matriks TF-IDF uji  :", X_test.shape)

## Pencarian k Optimal untuk KNN (GridSearchCV)

> **[PERBAIKAN]** Nilai k tidak lagi menggunakan heuristic √n, melainkan dicari secara empiris
> dengan 5-fold cross-validation. Metric `cosine` digunakan karena lebih cocok untuk TF-IDF sparse.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Kandidat nilai k yang akan diuji
k_candidates = [3, 5, 7, 9, 11, 15, 21, 31]

# GridSearchCV langsung pada X_train (sudah TF-IDF)
# metric='cosine' + algorithm='brute' karena sparse matrix
knn_grid = GridSearchCV(
    KNeighborsClassifier(metric='cosine', algorithm='brute'),
    param_grid={'n_neighbors': k_candidates},
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
knn_grid.fit(X_train, y_train)

best_k = knn_grid.best_params_['n_neighbors']
print(f"\nK terbaik    : {best_k}")
print(f"F1 CV terbaik: {knn_grid.best_score_ * 100:.2f}%")

# Tampilkan tabel hasil GridSearch
cv_results_df = pd.DataFrame({
    'k'      : k_candidates,
    'F1 CV'  : [knn_grid.cv_results_['mean_test_score'][i] * 100
                for i in range(len(k_candidates))],
    'Std'    : [knn_grid.cv_results_['std_test_score'][i] * 100
                for i in range(len(k_candidates))],
})
cv_results_df['F1 CV'] = cv_results_df['F1 CV'].apply(lambda x: f'{x:.2f}%')
cv_results_df['Std']   = cv_results_df['Std'].apply(lambda x: f'+/- {x:.2f}%')
display(cv_results_df)

## Klasifikasi

> **[PERBAIKAN]** KNN menggunakan `metric='cosine'` (bukan euclidean) dan `k` optimal dari GridSearchCV.

In [ ]:
# Model utama: KNN dengan cosine similarity dan k optimal
knn = KNeighborsClassifier(n_neighbors=best_k, metric='cosine', algorithm='brute')
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print(f"KNN (k={best_k}, cosine) — selesai dilatih")

In [ ]:
# Pembanding 1: Naive Bayes
nb_clf = MultinomialNB(alpha=0.1)
nb_clf.fit(X_train, y_train)
y_pred_nb = nb_clf.predict(X_test)
print("Naive Bayes — selesai dilatih")

In [ ]:
# Pembanding 2: Support Vector Machine
svm_clf = LinearSVC(class_weight='balanced', max_iter=5000, random_state=42)
svm_clf.fit(X_train, y_train)
y_pred_svm = svm_clf.predict(X_test)
print("Linear SVM — selesai dilatih")

## Evaluasi (Confusion Matrix)

Evaluasi memakai: (a) Confusion Matrix, (b) perhitungan manual TP/FP/FN/TN per kelas,
dan (c) ringkasan Accuracy, Precision, Recall, F1-Score.

In [ ]:
def manual_confusion_matrix(y_true, y_pred, labels):
    '''
    Hitung TP, FP, FN, TN per kelas secara MANUAL dari confusion matrix,
    persis seperti contoh tabel pada paper referensi.
    '''
    cm    = confusion_matrix(y_true, y_pred, labels=labels)
    total = cm.sum()
    rows  = []
    for i, label in enumerate(labels):
        TP = int(cm[i, i])
        FP = int(cm[:, i].sum() - TP)
        FN = int(cm[i, :].sum() - TP)
        TN = int(total - TP - FP - FN)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)
        rows.append({
            'Kelas'    : label,
            'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
            'Precision': round(precision, 4),
            'Recall'   : round(recall, 4),
            'F1-Score' : round(f1, 4),
        })
    return pd.DataFrame(rows)

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    '''Tampilkan confusion matrix, tabel manual TP/FP/FN/TN, dan ringkasan metrik.'''
    labels = sorted(pd.Series(y_true).unique())
    cm     = confusion_matrix(y_true, y_pred, labels=labels)

    print('=' * 60)
    print('MODEL:', model_name)
    print('=' * 60)

    # (a) Confusion matrix heatmap
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix — {model_name}')
    plt.tight_layout()
    plt.show()

    # (b) Tabel manual TP/FP/FN/TN per kelas
    print('Perhitungan Manual Confusion Matrix (per kelas):')
    tabel = manual_confusion_matrix(y_true, y_pred, labels)
    display(tabel)

    # (c) Ringkasan metrik keseluruhan (macro average)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f"Accuracy : {acc  * 100:.2f}%")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall   : {rec  * 100:.2f}%")
    print(f"F1-Score : {f1   * 100:.2f}%")
    return f1

### Evaluasi KNN (model utama paper — diperbaiki)

In [ ]:
f1_knn = evaluate_model(y_test, y_pred_knn, f'K-Nearest Neighbors (k={best_k}, cosine)')

### Evaluasi Naive Bayes (pembanding)

In [ ]:
f1_nb = evaluate_model(y_test, y_pred_nb, 'Naive Bayes')

### Evaluasi SVM (pembanding)

In [ ]:
f1_svm = evaluate_model(y_test, y_pred_svm, 'Support Vector Machine')

> Tabel manual menampilkan TP/FP/FN/TN tiap genre, lalu diringkas menjadi Accuracy, Precision, Recall, dan F1-Score.

## Analisis Tambahan

Validasi silang K-Fold, MSE & RMSE, dan perbandingan antar skenario preprocessing.

### Validasi Silang K-Fold (Stratified, k=5)

In [ ]:
X_cv  = df_balanced['synopsis_S3']
y_cv  = df_balanced['genre_main']
skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'Accuracy' : 'accuracy',
    'Precision': 'precision_macro',
    'Recall'   : 'recall_macro',
    'F1-Score' : 'f1_macro',
}

# [PERBAIKAN] KNN menggunakan cosine + k optimal dari GridSearchCV
cv_models = {
    'KNN (cosine, k optimal)': KNeighborsClassifier(
        n_neighbors=best_k, metric='cosine', algorithm='brute'),
    'Naive Bayes'            : MultinomialNB(alpha=0.1),
    'Linear SVM'             : LinearSVC(class_weight='balanced',
                                         max_iter=5000, random_state=42),
}

cv_rows = []
for name, model in cv_models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=20000, ngram_range=(1, 2),
            sublinear_tf=True, min_df=2, max_df=0.95)),
        ('clf', model),
    ])
    scores = cross_validate(pipe, X_cv, y_cv, cv=skf, scoring=scoring)
    row = {'Model': name}
    for label in scoring:
        mean = scores[f'test_{label}'].mean() * 100
        std  = scores[f'test_{label}'].std()  * 100
        row[label] = f"{mean:.2f}% +/- {std:.2f}%"
    cv_rows.append(row)

pd.DataFrame(cv_rows)

### Tabel MSE & RMSE

In [ ]:
le = LabelEncoder().fit(df_balanced['genre_main'])

def hitung_mse_rmse(y_true, y_pred):
    mse = mean_squared_error(le.transform(y_true), le.transform(y_pred))
    return mse, np.sqrt(mse)

mse_rows = []
for model_name, y_pred in [
    (f'KNN (k={best_k}, cosine)', y_pred_knn),
    ('Naive Bayes'              , y_pred_nb),
    ('Linear SVM'               , y_pred_svm),
]:
    mse, rmse = hitung_mse_rmse(y_test, y_pred)
    mse_rows.append({'Model': model_name, 'MSE': round(mse, 4), 'RMSE': round(rmse, 4)})

pd.DataFrame(mse_rows).sort_values('MSE').reset_index(drop=True)

### Perbandingan Antar Skenario Preprocessing (S1, S2, S3, S4)

> **[PERBAIKAN]** Ditambah skenario **S4** (title + synopsis + S3) dan KNN menggunakan cosine.

In [ ]:
scenarios = {
    'S1_stopword_only' : 'synopsis_S1',
    'S2_stemming_only' : 'synopsis_S2',
    'S3_stopword_stem' : 'synopsis_S3',
    'S4_title_synopsis': 'synopsis_S4',   # <- skenario baru
}
le_cmp  = LabelEncoder().fit(df_balanced['genre_main'])
results = []

for scen_name, text_col in scenarios.items():
    X_text_s = df_balanced[text_col]
    y_s      = df_balanced['genre_main']
    Xtr_t, Xte_t, ytr, yte = train_test_split(
        X_text_s, y_s, test_size=0.2, random_state=42, stratify=y_s)
    tfidf_s = TfidfVectorizer(
        max_features=20000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=2, max_df=0.95)
    Xtr = tfidf_s.fit_transform(Xtr_t)
    Xte = tfidf_s.transform(Xte_t)

    models_cmp = {
        'KNN (cosine)': KNeighborsClassifier(
            n_neighbors=best_k, metric='cosine', algorithm='brute'),
        'Naive Bayes' : MultinomialNB(alpha=0.1),
        'Linear SVM'  : LinearSVC(class_weight='balanced',
                                   max_iter=5000, random_state=42),
    }
    for model_name, model in models_cmp.items():
        model.fit(Xtr, ytr)
        yp  = model.predict(Xte)
        mse = mean_squared_error(le_cmp.transform(yte), le_cmp.transform(yp))
        results.append({
            'Scenario' : scen_name,
            'Model'    : model_name,
            'Accuracy' : accuracy_score(yte, yp),
            'Precision': precision_score(yte, yp, average='macro', zero_division=0),
            'Recall'   : recall_score(yte, yp, average='macro', zero_division=0),
            'F1'       : f1_score(yte, yp, average='macro', zero_division=0),
            'MSE'      : mse,
            'RMSE'     : np.sqrt(mse),
        })

results_df      = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
results_display = results_df.copy()
for c in ['Accuracy', 'Precision', 'Recall', 'F1']:
    results_display[c] = results_display[c].apply(lambda x: f"{x * 100:.2f}%")
for c in ['MSE', 'RMSE']:
    results_display[c] = results_display[c].apply(lambda x: f"{x:.4f}")
results_display

### Implementasi (Inferensi Genre)

Otomatis memakai kombinasi model + skenario dengan F1 tertinggi dari tabel perbandingan.

In [ ]:
best = results_df.iloc[0]   # sudah diurutkan descending F1
_prep = {
    'S1_stopword_only' : (True, False),
    'S2_stemming_only' : (False, True),
    'S3_stopword_stem' : (True, True),
    'S4_title_synopsis': (True, True),
}[best['Scenario']]

# Latih ulang model terbaik pada seluruh data balanced
final_tfidf = TfidfVectorizer(
    max_features=20000, ngram_range=(1, 2),
    sublinear_tf=True, min_df=2, max_df=0.95)

final_model = {
    'KNN (cosine)': KNeighborsClassifier(
        n_neighbors=best_k, metric='cosine', algorithm='brute'),
    'Naive Bayes' : MultinomialNB(alpha=0.1),
    'Linear SVM'  : LinearSVC(class_weight='balanced',
                               max_iter=5000, random_state=42),
}[best['Model']]

final_model.fit(
    final_tfidf.fit_transform(df_balanced[scenarios[best['Scenario']]]),
    df_balanced['genre_main']
)

def predict_genre(text, title=''):
    '''Prediksi genre dari sinopsis (dan opsional judul jika skenario S4).'''
    if best['Scenario'] == 'S4_title_synopsis':
        combined = title + ' ' + text
        vec = final_tfidf.transform([preprocess(combined, *_prep)])
    else:
        vec = final_tfidf.transform([preprocess(text, *_prep)])
    return final_model.predict(vec)[0]

print(f"Model terbaik  : {best['Model']} ({best['Scenario']}, F1={best['F1'] * 100:.2f}%)")
print()

# Uji beberapa contoh
contoh_list = [
    ("An open-world fantasy RPG where you level up your hero, learn magic spells, and battle epic bosses.",
     "Dragon Quest"),
    ("Build and manage your own farm. Plant crops, raise animals, and sell goods at the market.",
     "Farming Simulator"),
    ("A fast-paced shooter where you fight through waves of enemies using guns and explosives.",
     "Action Strike"),
    ("Explore mysterious dungeons, solve puzzles, and uncover ancient secrets in a rich narrative world.",
     "Mystery Expedition"),
]

for synopsis, title in contoh_list:
    pred = predict_genre(synopsis, title)
    print(f"Judul   : {title}")
    print(f"Sinopsis: {synopsis[:70]}...")
    print(f"Prediksi: {pred.upper()}")
    print()